# Vector Search for RAG

This notebook demonstrates using `RedisVectorStore` for retrieval-augmented generation (RAG).

**Features:**
- Store documents with vector embeddings
- Semantic similarity search
- Metadata filtering
- **Hybrid search** (vector + BM25 text search)
- Integration as agent tools

**Built on:** RedisVL's SearchIndex and VectorQuery

## Setup

Import dependencies and configure environment.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if OpenAI API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Please set it in .env file or environment"
    )

# Redis connection
REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Environment configured")
print(f"   Redis URL: {REDIS_URL}")

Environment configured
   Redis URL: redis://redis:6379


In [2]:
from agents import Agent, Runner, function_tool

print("OpenAI Agents SDK imported successfully")

OpenAI Agents SDK imported successfully


## Create Vector Store

Initialize a vector store for storing and searching documents.

In [3]:
from redis_openai_agents import RedisVectorStore

# Create a vector store for a knowledge base
store = RedisVectorStore(
    name="knowledge_base",
    redis_url=REDIS_URL,
)

print("RedisVectorStore created")
print(f"   Index name: {store.name}")

10:35:44 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:35:44 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


RedisVectorStore created
   Index name: knowledge_base


## Add Documents

Add documents to the vector store. Embeddings are generated automatically.

In [4]:
# Sample documents about Redis
documents = [
    {
        "content": "Redis is an open-source, in-memory data structure store that can be used as a database, cache, message broker, and streaming engine.",
        "metadata": {"source": "docs", "topic": "overview"}
    },
    {
        "content": "Redis supports data structures such as strings, hashes, lists, sets, sorted sets with range queries, bitmaps, hyperloglogs, geospatial indexes, and streams.",
        "metadata": {"source": "docs", "topic": "data-structures"}
    },
    {
        "content": "Redis has built-in replication, Lua scripting, LRU eviction, transactions, and different levels of on-disk persistence.",
        "metadata": {"source": "docs", "topic": "features"}
    },
    {
        "content": "Redis Cluster provides a way to run a Redis installation where data is automatically sharded across multiple Redis nodes.",
        "metadata": {"source": "docs", "topic": "clustering"}
    },
    {
        "content": "Redis 8 ships with vector search, JSON document storage, and time series capabilities built in.",
        "metadata": {"source": "docs", "topic": "redis"}
    },
]

# Add documents
ids = store.add_documents(documents)
print(f"Added {len(ids)} documents")
print(f"Document IDs: {ids}")

Added 5 documents
Document IDs: ['1568835c92f64dd2', 'c5a6d599b12e4969', '019548fa968b4806', '98593650c8094874', 'defd1f4d78ce4d07']


## Similarity Search

Search for documents similar to a query.

In [5]:
# Search for documents about data structures
query = "What data types does Redis support?"
results = store.search(query=query, k=3)

print(f"Query: {query}\n")
print(f"Found {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. Score: {result['score']:.4f}")
    print(f"   Content: {result['content'][:80]}...")
    print(f"   Metadata: {result['metadata']}\n")

Query: What data types does Redis support?

Found 3 results:

1. Score: 0.7517
   Content: Redis supports data structures such as strings, hashes, lists, sets, sorted sets...
   Metadata: {'source': 'docs', 'topic': 'data-structures'}

2. Score: 0.7229
   Content: Redis has built-in replication, Lua scripting, LRU eviction, transactions, and d...
   Metadata: {'source': 'docs', 'topic': 'features'}

3. Score: 0.6367
   Content: Redis is an open-source, in-memory data structure store that can be used as a da...
   Metadata: {'source': 'docs', 'topic': 'overview'}



## Filtered Search

Search with metadata filters to narrow results.

In [6]:
# Search only in documents about features
query = "What capabilities does Redis have?"
results = store.search(
    query=query,
    k=3,
    filter={"topic": "features"}
)

print(f"Query: {query}")
print(f"Filter: topic=features\n")

for i, result in enumerate(results, 1):
    print(f"{i}. Score: {result['score']:.4f}")
    print(f"   Content: {result['content'][:80]}...")
    print(f"   Topic: {result['metadata'].get('topic')}\n")

Query: What capabilities does Redis have?
Filter: topic=features

1. Score: 0.7311
   Content: Redis has built-in replication, Lua scripting, LRU eviction, transactions, and d...
   Topic: features



## Hybrid Search (Vector + BM25)

Combine semantic vector search with BM25 keyword matching for best results.
Uses Reciprocal Rank Fusion (RRF) to merge results from both methods.

In [7]:
# Hybrid search combines vector similarity with BM25 text matching
query = "Redis data structures strings hashes"

# Default: equal weight for vector and text
results = store.hybrid_search(query=query, k=3)

print(f"Query: {query}")
print(f"Hybrid search (50% vector, 50% text):\n")

for i, result in enumerate(results, 1):
    print(f"{i}. Score: {result['score']:.6f}")
    print(f"   Content: {result['content'][:80]}...")
    print()

Query: Redis data structures strings hashes
Hybrid search (50% vector, 50% text):

1. Score: 0.016667
   Content: Redis supports data structures such as strings, hashes, lists, sets, sorted sets...

2. Score: 0.008197
   Content: Redis is an open-source, in-memory data structure store that can be used as a da...

3. Score: 0.008065
   Content: Redis has built-in replication, Lua scripting, LRU eviction, transactions, and d...



In [8]:
# Weighted hybrid search - favor keyword matching
print("Heavy text weight (90% text, 10% vector):")
results_text = store.hybrid_search(
    query=query,
    k=3,
    text_weight=0.9,
    vector_weight=0.1,
)

for i, result in enumerate(results_text, 1):
    print(f"{i}. Score: {result['score']:.6f}")
    print(f"   Content: {result['content'][:60]}...")
    print()

print("\nHeavy vector weight (10% text, 90% vector):")
results_vector = store.hybrid_search(
    query=query,
    k=3,
    text_weight=0.1,
    vector_weight=0.9,
)

for i, result in enumerate(results_vector, 1):
    print(f"{i}. Score: {result['score']:.6f}")
    print(f"   Content: {result['content'][:60]}...")

Heavy text weight (90% text, 10% vector):


1. Score: 0.016667
   Content: Redis supports data structures such as strings, hashes, list...

2. Score: 0.001639
   Content: Redis is an open-source, in-memory data structure store that...

3. Score: 0.001613
   Content: Redis has built-in replication, Lua scripting, LRU eviction,...


Heavy vector weight (10% text, 90% vector):


1. Score: 0.016667
   Content: Redis supports data structures such as strings, hashes, list...
2. Score: 0.014754
   Content: Redis is an open-source, in-memory data structure store that...
3. Score: 0.014516
   Content: Redis has built-in replication, Lua scripting, LRU eviction,...


## Integration with Agents

Create a tool that lets agents search the knowledge base.

In [9]:
@function_tool
def search_knowledge_base(query: str, num_results: int = 3) -> str:
    """
    Search the Redis knowledge base for relevant information.
    
    Args:
        query: The search query
        num_results: Number of results to return (default 3)
        
    Returns:
        Formatted search results
    """
    results = store.search(query=query, k=num_results)
    
    if not results:
        return "No relevant documents found."
    
    formatted = []
    for i, r in enumerate(results, 1):
        formatted.append(f"{i}. {r['content']}")
    
    return "\n\n".join(formatted)

print("Tool created: search_knowledge_base")

Tool created: search_knowledge_base


In [10]:
# Create an agent with access to the knowledge base
rag_agent = Agent(
    name="redis_expert",
    instructions="""You are a Redis expert assistant. 
    Use the search_knowledge_base tool to find relevant information before answering questions.
    Base your answers on the search results.""",
    tools=[search_knowledge_base]
)

print("Agent created with knowledge base access")

Agent created with knowledge base access


In [11]:
# Ask the agent a question
question = "How can I scale Redis across multiple servers?"
print(f"User: {question}\n")

result = await Runner.run(rag_agent, input=question)
print(f"Assistant: {result.final_output}")

User: How can I scale Redis across multiple servers?



10:35:49 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


10:35:53 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Assistant: To scale Redis across multiple servers, you can use Redis Cluster. Redis Cluster allows you to automatically shard data across several Redis nodes. This setup enables your Redis deployment to handle more data and requests than a single server could manage by distributing the load and dataset.

Key points about scaling with Redis Cluster:
- Data is automatically partitioned (sharded) among multiple nodes.
- Redis Cluster can continue to operate even if some nodes fail, providing higher availability.
- You interact with the cluster as a single logical database, while behind the scenes, data is managed across multiple Redis servers.

Additionally, Redis supports replication, which can be used alongside sharding for redundancy and read scalability. Combining clusters and replication, you can scale Redis both in terms of data capacity and resilience.


## Document Count and Stats

In [12]:
# Get document count
count = store.count()
print(f"Documents in store: {count}")

Documents in store: 5


## Inspect in Redis

Use RedisVL CLI to inspect the vector index.

In [13]:
!rvl index listall -u $REDIS_URL

10:35:53 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL


10:35:53 [RedisVL] INFO   Indices:
10:35:53 [RedisVL] INFO   1. test_hybrid_filter
10:35:53 [RedisVL] INFO   2. knowledge_base


In [14]:
!rvl index info -i knowledge_base -u $REDIS_URL

10:35:53 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL


Index Information:
╭─────────────────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┬╮
│ Index Name              │ Storage Type            │ Prefixes                │ Index Options           │ Indexing                │
├─────────────────────────┼─────────────────────────┼─────────────────────────┼─────────────────────────┼─────────────────────────┼┤
| knowledge_base          | HASH                    | ['doc:knowledge_base:'] | []                      | 0                       |
╰─────────────────────────┴─────────────────────────┴─────────────────────────┴─────────────────────────┴─────────────────────────┴╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────

## Delete Documents

In [15]:
# Delete all documents and the index
store.delete_all()
print(f"Deleted all documents")
print(f"Documents remaining: {store.count()}")

Deleted all documents
Documents remaining: 0


## Summary

This notebook demonstrated:

1. **Vector Store Creation** - initialize with index name and Redis URL
2. **Document Storage** - add documents with automatic embedding generation
3. **Similarity Search** - find documents semantically similar to queries
4. **Filtered Search** - narrow results using metadata filters
5. **Hybrid Search** - combine vector + BM25 text search with configurable weights
6. **Agent Integration** - use vector search as a tool for RAG

**Key Benefits:**
- Semantic search finds relevant documents even with different wording
- Hybrid search combines semantic understanding with keyword matching
- Metadata filtering enables scoped searches
- Easy integration with OpenAI Agents SDK

**Next Steps:**
- Add more documents to your knowledge base
- Experiment with hybrid search weights for your use case
- Combine with SemanticCache for caching RAG results